# 1- Leitura e Padronização

## Carregando (leitura) da  imagem 

In [8]:
import cv2 as cv
def mostrar_imagem(img):
    cv.imshow("Amostra da imagem",img)
    cv.waitKey(0) #tempo de leitura
    cv.destroyAllWindows()

In [ ]:
#importando bibliotecas
#Leia a imagem original do disco.
#Redimensione a imagem para um tamanho padrão (por exemplo, 800x600) para facilitar o processamento.

import cv2 as cv
import numpy as np

#definindo variaveis
largura=800
altura=600
pts_destino=np.float32([
    [0,0], #canto superior esquerdo
    [largura,0], #canto superior direito
    [largura,altura], #canto inferior direito
    [0,altura] #canto inferior esquerdo
])


def padronizar(caminho_imagem):

    #LEITURA DA IMAGEM
    img=cv.imread(caminho_imagem)
    
    if img is not None:
        mostrar_imagem(img)
        print("imagem carregada")
        return cv.resize(img,(altura,largura))
    else:
        print("erro ao carregar imagem")

'''2. Correção de Perspectiva (Envelopamento):
2.1 Defina os 4 pontos de origem (os cantos distorcidos da bandeja na foto) e os 4 pontos de destino (os cantos de um retângulo perfeito).
2.2 Utilize as funções matemáticas do OpenCV para realizar o warp perspective (envelopamento), gerando uma nova imagem onde a bandeja aparece perfeitamente vista de cima. '''

def _ordenar_pontos(pontos):
    '''ordena pontos'''
    rect = np.zeros((4, 2), dtype="float32")
    s = pontos.sum(axis=1)
    rect[0] = pontos[np.argmin(s)]
    rect[2] = pontos[np.argmax(s)]
    diff = np.diff(pontos, axis=1)
    rect[1] = pontos[np.argmin(diff)]
    rect[3] = pontos[np.argmax(diff)]
    return rect

def detectar_cantos(img_resized):
    '''detecta os cantos na imagem original'''
    img_gray=cv.cvtColor(img_resized,cv.COLOR_BGR2GRAY)
    #img blur
    img_blur=cv.GaussianBlur(img_gray,[11,11],0)
    ## threshold
    _,mask=cv.threshold(img_blur,0,255,cv.THRESH_BINARY + cv.THRESH_OTSU)
    mostrar_imagem(mask)

    # Encontra o maior contorno (que será a folha de papel)
    contornos, _ = cv.findContours(mask, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)
    contorno_maior = max(contornos, key=cv.contourArea)
        
    perimetro = cv.arcLength(contorno_maior, True)
    aproximacao = cv.approxPolyDP(contorno_maior, 0.02 * perimetro, True)
        
    if len(aproximacao) == 4:
        return _ordenar_pontos(aproximacao.reshape(4, 2))
            
    # Fallback se algo der muito errado
    print("Aviso: Papel não detectado. Usando pontos de fallback.")
    return np.float32([[100, 150], [700, 150], [50, 500], [750, 500]])

def corrigir_perspectiva(img_resized, pts_origem):
    # Realiza o warp perspective (envelopamento) com os 4 pontos detectados[cite: 1]
    matriz = cv.getPerspectiveTransform(pts_origem, pts_destino)
    return cv.warpPerspective(img_resized, matriz, (largura, altura))
    
def _pre_processar(img_warped):
        '''
"""
3. Pré-processamento e Mudança de Cor:
-Converta a imagem "envelopada" de BGR para Tons de Cinza.
-Aplique um desfoque (Blur) e um limiar (Threshold) ou Canny para criar uma máscara binária (fundo preto e objetos brancos)."""
'''
        """AJUSTE 2: Inversão de cores para atender ao PDF.
        """
        # Converte a imagem envelopada para Tons de Cinza[cite: 1]
    img_gray = cv.cvtColor(img_warped, cv.COLOR_BGR2GRAY)
        
        # Aplica um desfoque suave para remover ruídos na superfície das peças[cite: 1]
    img_blur = cv.GaussianBlur(img_gray, (5, 5), 0)
        
        # Aplica limiar para criar máscara binária (fundo preto e objetos brancos)[cite: 1]
        # Como o fundo é branco e as peças são escuras, usamos THRESH_BINARY_INV com Otsu.
    _, mask = cv.threshold(img_blur, 0, 255, cv.THRESH_BINARY_INV + cv.THRESH_OTSU)
    mostrar_imagem(mask)
    return img_gray, mask

In [10]:
caminho_imagem="data/raw/Exer_1.jpeg"
img_padronizada=padronizar(caminho_imagem)
pts_origem=detectar_cantos(img_padronizada)
img_fix_perspectiva=corrigir_perspectiva(img_padronizada,pts_origem)

mostrar_imagem(img_fix_perspectiva)



#mostrar_imagem(img_padronizada)

imagem carregada
Aviso: Papel não detectado. Usando pontos de fallback.
